In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Splitting into Training and Testing set
X = pd.read_csv("../data/processed/cicids2017_features_reduced.csv")
y = pd.read_csv("../data/processed/cicids2017_labels.csv").squeeze()

print(X.shape, y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)

(610492, 46) (610492,)
Train: (488393, 46) Test: (122099, 46)


In [3]:
# Scale features

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
# Set up evaluation

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)

import time
def evaluate_models(name, model, X_test, y_test):
    start = time.time()
    y_pred = model.predict(X_test)
    inference_time = time.time()-start

    y_prob = model.predict_prob(X_test)[:, 1] if hasattr(model, "predict_prob") else y_pred

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr = fp / (fp + tn)

    print(f"\n--- {name} ---")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred): .4f}")
    print(f"Precision: {precision_score(y_test, y_pred): .4f}")
    print(f"Recall:    {recall_score(y_test, y_pred): .4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred): .4f}")
    print(f"ROC_AUC:   {roc_auc_score(y_test, y_prob): .4f}")
    print(f"FPR:       {fpr:.4f}")
    print(f"Inference time({len(X_test)} samples:) {inference_time: .4f}s")

    return{
        "model": name, "accuracy":accuracy_score(y_test, y_pred), 
        "precision": precision_score(y_test, y_pred), "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred), "roc-auc": roc_auc_score(y_test, y_prob),
        "fpr": fpr, "inference_time": inference_time
    }

results_df = pd.read_csv("../data/processed/baseline_results.csv")
results = results_df.to_dict('records')

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=None, min_samples_split=5, random_state=42)
rf.fit(X_train_scaled, y_train)
results.append(evaluate_models("Random Forest", rf, X_test_scaled, y_test))


--- Random Forest ---
Accuracy:   0.9997
Precision:  0.9994
Recall:     0.9997
F1-score:   0.9995
ROC_AUC:    0.9997
FPR:       0.0003
Inference time(122099 samples:)  1.2937s


In [7]:
from xgboost import XGBClassifier
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb.fit(X_train_scaled, y_train)
results.append(evaluate_models("XGBoost", xgb, X_test_scaled, y_test))


--- XGBoost ---
Accuracy:   0.9999
Precision:  0.9996
Recall:     0.9999
F1-score:   0.9998
ROC_AUC:    0.9999
FPR:       0.0002
Inference time(122099 samples:)  0.1327s


In [8]:
results_df = pd.DataFrame(results)
print(results_df)
results_df.to_csv("../data/processed/all_models_results.csv", index=False)


                model  accuracy  precision    recall        f1   roc-auc  \
0  LogisticRegression  0.990803   0.987687  0.983278  0.985478  0.988789   
1                 KNN  0.998993   0.997886  0.998942  0.998414  0.998979   
2       Decision Tree  0.998043   0.997725  0.996103  0.996914  0.997524   
3       Random Forest  0.999689   0.999355  0.999665  0.999510  0.999682   
4             XGBoost  0.999853   0.999639  0.999897  0.999768  0.999864   

        fpr  inference_time  
0  0.005699        0.020446  
1  0.000984      114.182894  
2  0.001056        0.103908  
3  0.000300        1.293651  
4  0.000168        0.132722  


In [9]:
import joblib

joblib.dump(rf, "../models/random_forest.pkl")
joblib.dump(xgb, "../models/xgboost.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']